# 🏍️ ATVED - Train Custom Helmet Detection Model

This notebook fine-tunes YOLOv8 to detect:
- `helmet` - rider wearing a helmet
- `no-helmet` - rider NOT wearing a helmet  
- `head` - visible head (for cross-checking)

**Runtime:** Make sure you're using a **GPU runtime**:  
`Runtime → Change runtime type → T4 GPU`

**Training time:** ~30-60 minutes on T4 GPU

## Step 1: Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ NO GPU DETECTED! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Download Helmet Detection Dataset from Roboflow

We use a public helmet detection dataset from Roboflow Universe.

### How to get your FREE Roboflow API key:
1. Go to [roboflow.com](https://roboflow.com) and sign up (free)
2. Go to **Settings → API Key**
3. Copy your API key and paste it below

In [ ]:
# ===== PASTE YOUR ROBOFLOW API KEY HERE =====
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- Replace this!
# =============================================

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Download a high-quality helmet detection dataset
# This dataset has ~5000 images with helmet/no-helmet/head classes
project = rf.workspace("joseph-nelson").project("helmet-detection-iosqr")
version = project.version(2)
dataset = version.download("yolov8")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

### Alternative: If the above dataset is unavailable, try this one instead
Uncomment and run the cell below ONLY if Step 2 above failed.

In [ ]:
# # ALTERNATIVE DATASET (uncomment if needed)
# project = rf.workspace("accident-detection-ffdrf").project("helmet-detection-yolov8")
# version = project.version(1)
# dataset = version.download("yolov8")
# print(f"\n✅ Dataset downloaded to: {dataset.location}")

## Step 3: Verify Dataset

In [ ]:
import os
import yaml

# Find and display the data.yaml
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📋 Dataset Configuration:")
print(f"   Classes: {data_config.get('names', data_config.get('nc', 'unknown'))}")
print(f"   Number of classes: {data_config.get('nc', 'unknown')}")

# Count images
train_dir = os.path.join(dataset.location, "train", "images")
val_dir = os.path.join(dataset.location, "valid", "images")

train_count = len(os.listdir(train_dir)) if os.path.exists(train_dir) else 0
val_count = len(os.listdir(val_dir)) if os.path.exists(val_dir) else 0

print(f"   Training images: {train_count}")
print(f"   Validation images: {val_count}")
print(f"\n✅ Dataset is ready for training!")

## Step 4: Train YOLOv8 🚀

This is the main training step. It will take ~30-60 minutes on a T4 GPU.

We start from pre-trained YOLOv8n weights and fine-tune on our helmet data.

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Train on our helmet dataset
results = model.train(
    data=data_yaml_path,
    epochs=50,           # 50 epochs is a good balance of speed vs accuracy
    imgsz=640,           # Standard input size
    batch=16,            # Batch size (T4 can handle 16)
    name="atved_helmet", # Output folder name
    patience=10,         # Stop early if no improvement for 10 epochs
    save=True,           # Save checkpoints
    plots=True,          # Generate training plots
    verbose=True,
)

print("\n" + "="*60)
print("  ✅ TRAINING COMPLETE!")
print("="*60)

## Step 5: Evaluate the Model

In [ ]:
# Load the best model from training
best_model_path = "runs/detect/atved_helmet/weights/best.pt"
trained_model = YOLO(best_model_path)

# Run validation
metrics = trained_model.val()

print("\n📊 Model Performance:")
print(f"   mAP50:     {metrics.box.map50:.3f}")
print(f"   mAP50-95:  {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall:    {metrics.box.mr:.3f}")

if metrics.box.map50 > 0.7:
    print("\n  ✅ Model accuracy is GOOD! Ready for deployment.")
elif metrics.box.map50 > 0.5:
    print("\n  ⚠️ Model accuracy is OK. Consider training for more epochs.")
else:
    print("\n  ❌ Model accuracy is LOW. Try a larger dataset or more epochs.")

## Step 6: View Training Plots

In [ ]:
from IPython.display import Image, display
import os

plots_dir = "runs/detect/atved_helmet"

# Show confusion matrix
cm_path = os.path.join(plots_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    print("Confusion Matrix:")
    display(Image(filename=cm_path, width=600))

# Show training results
results_path = os.path.join(plots_dir, "results.png")
if os.path.exists(results_path):
    print("\nTraining Curves:")
    display(Image(filename=results_path, width=800))

# Show sample predictions
pred_path = os.path.join(plots_dir, "val_batch0_pred.png")
if os.path.exists(pred_path):
    print("\nSample Predictions on Validation Set:")
    display(Image(filename=pred_path, width=800))

## Step 7: Test on a Sample Image

In [ ]:
import glob

# Find a validation image to test on
val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*"))

if val_images:
    test_img = val_images[0]
    print(f"Testing on: {test_img}")
    
    results = trained_model.predict(test_img, conf=0.3, save=True)
    
    # Show the prediction
    pred_dir = "runs/detect/predict"
    pred_images = glob.glob(os.path.join(pred_dir, "*"))
    if pred_images:
        display(Image(filename=pred_images[0], width=600))
    
    # Print detections
    for r in results:
        for box in r.boxes:
            cls_name = trained_model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            print(f"  Detected: {cls_name} ({conf:.0%})")
else:
    print("No validation images found.")

## Step 8: Download Your Trained Model! 📥

Run the cell below to download `best.pt` to your computer.  
Then place it in your project folder:
```
Gridlock_sunny/models/atved_helmet_best.pt
```

In [ ]:
from google.colab import files
import shutil

# Copy the best model to a clean filename
src = "runs/detect/atved_helmet/weights/best.pt"
dst = "atved_helmet_best.pt"
shutil.copy(src, dst)

# Also copy the last model as backup
src_last = "runs/detect/atved_helmet/weights/last.pt"
dst_last = "atved_helmet_last.pt"
shutil.copy(src_last, dst_last)

print(f"✅ Model saved as: {dst}")
print(f"   Backup saved as: {dst_last}")
print(f"\n📥 Downloading to your computer...")

# Trigger browser download
files.download(dst)

print("\n" + "="*60)
print("  DONE! Place the downloaded .pt file in:")
print("  Gridlock_sunny/models/atved_helmet_best.pt")
print("="*60)